# MVP - Mapa de Calor de Entregas (Vitória - ES)

## 1. Importação de Bibliotecas

In [7]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
import folium
from folium.plugins import HeatMap

## 2. Simulação de Dados

Coordenadas fixadas em **áreas urbanas densas** de Vitória - ES, sem matas nem água.

| Zona | Bairro | Característica |
|------|--------|----------------|
| A | Jardim da Penha | Bairro residencial/comercial denso, nordeste da ilha |
| B | Praia do Canto / Enseada do Suá | Área nobre com alta densidade residencial |
| C | Bento Ferreira / Santa Lúcia | Bairros centrais com muitas farmácias e comércios |

In [8]:
np.random.seed(42)
n_pedidos = 300
zonas = [
    # Jardim da Penha: miolo do bairro, longe da praia e do manguezal
    {"lat": -20.2810, "lon": -40.3020, "std": 0.0018, "peso": 0.40},
    # Praia do Canto / Enseada do Suá: interior das quadras, longe da orla
    {"lat": -20.3000, "lon": -40.2980, "std": 0.0015, "peso": 0.30},
    # Bento Ferreira / Santa Lúcia: centro residencial, sem matas
    {"lat": -20.3060, "lon": -40.3180, "std": 0.0018, "peso": 0.30},
]

latitudes  = np.concatenate([np.random.normal(z["lat"], z["std"], int(n_pedidos * z["peso"])) for z in zonas])
longitudes = np.concatenate([np.random.normal(z["lon"], z["std"], int(n_pedidos * z["peso"])) for z in zonas])

df_pedidos = pd.DataFrame({
    'pedido_id'     : range(1, n_pedidos + 1),
    'latitude'      : latitudes,
    'longitude'     : longitudes,
    'valor_entrega' : np.random.uniform(7.00, 18.00, n_pedidos)
})

print(f"Total de pedidos: {len(df_pedidos)}")
print(f"Lat: {latitudes.min():.4f} → {latitudes.max():.4f}")
print(f"Lon: {longitudes.min():.4f} → {longitudes.max():.4f}")
df_pedidos.head()

Total de pedidos: 300
Lat: -20.3118 → -20.2766
Lon: -40.3224 → -40.2934


,pedido_id,latitude,longitude,valor_entrega
0,1,-20.280106,-40.303492,9.521765
1,2,-20.281249,-40.303008,14.944778
2,3,-20.279834,-40.300655,14.920402
3,4,-20.278259,-40.300901,14.052624
4,5,-20.281421,-40.302038,14.633433


## 3. Machine Learning: Identificação de Hotspots (K-Means)

In [9]:
X = df_pedidos[['latitude', 'longitude']]
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X)
df_pedidos['cluster_hotspot'] = kmeans.labels_

hotspots_centros  = kmeans.cluster_centers_
contagem_cluster  = df_pedidos['cluster_hotspot'].value_counts().sort_index()

print("Hotspots detectados:")
for i, c in enumerate(hotspots_centros):
    print(f"  Hotspot {i+1}: ({c[0]:.4f}, {c[1]:.4f})  |  {contagem_cluster[i]} pedidos")

Hotspots detectados:
  Hotspot 1: (-20.2811, -40.3019)  |  120 pedidos
  Hotspot 2: (-20.3060, -40.3182)  |  90 pedidos
  Hotspot 3: (-20.2999, -40.2981)  |  90 pedidos


c:\Users\Usuario\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(


## 4. Mapa de Calor Interativo — Módulo do Entregador

In [10]:
# Centro do mapa: ponto médio entre os bairros urbanos usados
mapa_vitoria = folium.Map(
    location=[-20.2950, -40.3060],
    zoom_start=14,
    tiles="OpenStreetMap"
)

# HeatMap
dados_calor = df_pedidos[['latitude', 'longitude']].values.tolist()
HeatMap(
    dados_calor,
    radius=22,
    blur=14,
    min_opacity=0.45,
    max_zoom=16,
    gradient={0.2: 'blue', 0.45: 'lime', 0.75: 'orange', 1.0: 'red'}
).add_to(mapa_vitoria)

# Marcadores dos hotspots
cores       = ["red", "blue", "green"]
nomes_zonas = ["Jardim da Penha", "Praia do Canto / Enseada do Suá", "Bento Ferreira / Santa Lúcia"]

for i, centro in enumerate(hotspots_centros):
    folium.Marker(
        location=[centro[0], centro[1]],
        popup=folium.Popup(
            f"<b>🔥 Hotspot {i+1} — {nomes_zonas[i]}</b><br>"
            f"📦 Pedidos na área: {contagem_cluster[i]}<br>"
            f"📍 ({centro[0]:.4f}, {centro[1]:.4f})",
            max_width=260
        ),
        tooltip=f"Hotspot {i+1}: {nomes_zonas[i]}",
        icon=folium.Icon(color=cores[i], icon="fire", prefix="fa")
    ).add_to(mapa_vitoria)

folium.LayerControl().add_to(mapa_vitoria)
mapa_vitoria

## 5. Exportar Mapa como HTML

In [11]:
mapa_vitoria.save("mapa_de_calor_entregador.html")
print("MVP Gerado: 'mapa_de_calor_entregador.html' salvo com sucesso!")

MVP Gerado: 'mapa_de_calor_entregador.html' salvo com sucesso!
